# Labwork 2 - Supervised learning

This notebook runs k-nearest neighbors classification on two labelled datasets:

- `Real estate.csv`, using `Y house price of unit area` as the label
- `song_data.csv`, using `song_popularity` as the label

Both target columns are numeric, so they are converted into three classes using quantile bins: `low`, `medium`, and `high`. This makes the task suitable for classification metrics: classification error, precision, recall, and F1-score.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

RANDOM_STATE = 42
K_VALUES = [1, 3, 5, 7, 9, 11, 15, 21]

## Helper functions

In [2]:
def make_quantile_classes(series, labels=("low", "medium", "high")):
    """Convert a numeric target into balanced classification labels."""
    return pd.qcut(series, q=len(labels), labels=labels, duplicates="drop")


def evaluate_knn(X, y, k_values=K_VALUES, normalize=False):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y
    )

    rows = []
    fitted_models = {}

    for k in k_values:
        if normalize:
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("knn", KNeighborsClassifier(n_neighbors=k))
            ])
        else:
            model = KNeighborsClassifier(n_neighbors=k)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        rows.append({
            "k": k,
            "classification_error": 1 - accuracy,
            "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
            "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
        })
        fitted_models[k] = (model, y_test, y_pred)

    results = pd.DataFrame(rows)
    best_k = results.sort_values(["f1_score", "classification_error"], ascending=[False, True]).iloc[0]["k"]
    return results, int(best_k), fitted_models[int(best_k)]


def print_best_model_details(dataset_name, best_k, model_data):
    model, y_test, y_pred = model_data
    print(f"{dataset_name} - best k: {best_k}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion matrix:")
    display(pd.DataFrame(
        confusion_matrix(y_test, y_pred, labels=sorted(y_test.unique())),
        index=[f"actual_{label}" for label in sorted(y_test.unique())],
        columns=[f"pred_{label}" for label in sorted(y_test.unique())]
    ))

def evaluate_model_family(X, y, model_name, normalize=False):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y
    )

    if model_name == "Logistic Regression":
        candidates = [
            ("C=0.1", LogisticRegression(C=0.1, max_iter=2000, random_state=RANDOM_STATE)),
            ("C=1", LogisticRegression(C=1, max_iter=2000, random_state=RANDOM_STATE)),
            ("C=10", LogisticRegression(C=10, max_iter=2000, random_state=RANDOM_STATE)),
        ]
    elif model_name == "SVM":
        candidates = [
            ("C=0.1", LinearSVC(C=0.1, max_iter=10000, random_state=RANDOM_STATE)),
            ("C=1", LinearSVC(C=1, max_iter=10000, random_state=RANDOM_STATE)),
            ("C=10", LinearSVC(C=10, max_iter=10000, random_state=RANDOM_STATE)),
        ]
    else:
        raise ValueError("model_name must be 'Logistic Regression' or 'SVM'")

    rows = []
    fitted_models = {}

    for setting, estimator in candidates:
        if normalize:
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("model", estimator)
            ])
        else:
            model = estimator

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        rows.append({
            "model": model_name,
            "setting": setting,
            "classification_error": 1 - accuracy,
            "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
            "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
        })
        fitted_models[setting] = (model, y_test, y_pred)

    results = pd.DataFrame(rows)
    best_setting = results.sort_values(["f1_score", "classification_error"], ascending=[False, True]).iloc[0]["setting"]
    return results, best_setting, fitted_models[best_setting]


## Dataset 1: Real estate

The numeric house price target is converted to three classes. The `No` column is an identifier, so it is not used as an input feature.

In [3]:
real_estate = pd.read_csv("Real estate.csv")
real_estate.head()

,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1


In [4]:
real_estate_target = "Y house price of unit area"

real_estate_y = make_quantile_classes(real_estate[real_estate_target])
real_estate_X = real_estate.drop(columns=["No", real_estate_target])

print("Class distribution:")
display(real_estate_y.value_counts().sort_index())

print("Input features:")
display(real_estate_X.head())

Class distribution:


Y house price of unit area
low       141
medium    136
high      137
Name: count, dtype: int64

Input features:


,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude
0,2012.917,32.0,84.87882,10,24.98298,121.54024
1,2012.917,19.5,306.59470,9,24.98034,121.53951
2,2013.583,13.3,561.98450,5,24.98746,121.54391
3,2013.500,13.3,561.98450,5,24.98746,121.54391
4,2012.833,5.0,390.56840,5,24.97937,121.54245


## Task 1: Run k-NN on the Real Estate Dataset

This section trains k-nearest neighbors classifiers on the real estate features using the binned `Y house price of unit area` class label.

## Task 2 and Task 3: Real Estate Metrics and Different k Values

The output table compares the predicted test labels with the original test labels for several values of `k`. It reports classification error, precision, recall, and F1-score.

In [5]:
real_results, real_best_k, real_best_model = evaluate_knn(real_estate_X, real_estate_y, normalize=False)
real_results

,k,classification_error,precision,recall,f1_score
0,1,0.259615,0.743967,0.740385,0.741758
1,3,0.221154,0.789798,0.778846,0.778321
2,5,0.201923,0.802642,0.798077,0.793862
3,7,0.192308,0.807249,0.807692,0.800063
4,9,0.201923,0.798866,0.798077,0.791106
5,11,0.182692,0.821583,0.817308,0.811644
6,15,0.182692,0.821583,0.817308,0.811644
7,21,0.240385,0.755516,0.759615,0.753614


### Task 3 Comment: Real Estate k Values

When `k` changes, the real estate performance first improves and then gets worse. The best unnormalized result is at `k=11` or `k=15`, with classification error about `0.183` and weighted F1-score about `0.812`. `k=1` performs worse because it is too sensitive to individual training samples. Very large `k`, such as `k=21`, also performs worse because the prediction becomes too smooth and ignores local patterns.

## Task 4: Normalize the Real Estate Dataset

This section repeats k-NN after applying `StandardScaler` to the input features.

In [6]:
real_scaled_results, real_scaled_best_k, real_scaled_best_model = evaluate_knn(real_estate_X, real_estate_y, normalize=True)
real_scaled_results

,k,classification_error,precision,recall,f1_score
0,1,0.259615,0.744926,0.740385,0.735243
1,3,0.259615,0.744893,0.740385,0.734975
2,5,0.259615,0.742419,0.740385,0.730894
3,7,0.250000,0.745614,0.750000,0.743902
4,9,0.250000,0.747315,0.750000,0.741868
5,11,0.250000,0.753011,0.750000,0.744795
6,15,0.250000,0.753011,0.750000,0.744795
7,21,0.240385,0.761588,0.759615,0.754352


## Real Estate Normalized vs. Unnormalized Comparison

The table below places both versions side by side so the effect of normalization can be compared directly.

In [7]:
real_comparison = real_results.merge(
    real_scaled_results,
    on="k",
    suffixes=("_without_normalization", "_with_normalization")
)
real_comparison

,k,classification_error_without_normalization,precision_without_normalization,recall_without_normalization,f1_score_without_normalization,classification_error_with_normalization,precision_with_normalization,recall_with_normalization,f1_score_with_normalization
0,1,0.259615,0.743967,0.740385,0.741758,0.259615,0.744926,0.740385,0.735243
1,3,0.221154,0.789798,0.778846,0.778321,0.259615,0.744893,0.740385,0.734975
2,5,0.201923,0.802642,0.798077,0.793862,0.259615,0.742419,0.740385,0.730894
3,7,0.192308,0.807249,0.807692,0.800063,0.250000,0.745614,0.750000,0.743902
4,9,0.201923,0.798866,0.798077,0.791106,0.250000,0.747315,0.750000,0.741868
5,11,0.182692,0.821583,0.817308,0.811644,0.250000,0.753011,0.750000,0.744795
6,15,0.182692,0.821583,0.817308,0.811644,0.250000,0.753011,0.750000,0.744795
7,21,0.240385,0.755516,0.759615,0.753614,0.240385,0.761588,0.759615,0.754352


## Real Estate Best Model Details

The classification reports and confusion matrices show the detailed test-set performance for the best unnormalized and normalized real estate models.

In [8]:
print_best_model_details("Real estate without normalization", real_best_k, real_best_model)
print_best_model_details("Real estate with normalization", real_scaled_best_k, real_scaled_best_model)

Real estate without normalization - best k: 11

Classification report:
              precision    recall  f1-score   support

        high       0.75      0.88      0.81        34
         low       0.87      0.94      0.91        36
      medium       0.84      0.62      0.71        34

    accuracy                           0.82       104
   macro avg       0.82      0.81      0.81       104
weighted avg       0.82      0.82      0.81       104

Confusion matrix:


,pred_high,pred_low,pred_medium
actual_high,30,1,3
actual_low,1,34,1
actual_medium,9,4,21


Real estate with normalization - best k: 21

Classification report:
              precision    recall  f1-score   support

        high       0.67      0.82      0.74        34
         low       0.92      0.92      0.92        36
      medium       0.69      0.53      0.60        34

    accuracy                           0.76       104
   macro avg       0.76      0.76      0.75       104
weighted avg       0.76      0.76      0.75       104

Confusion matrix:


,pred_high,pred_low,pred_medium
actual_high,28,1,5
actual_low,0,33,3
actual_medium,14,2,18


### Real estate comments

For the real estate dataset, the best unnormalized result is at `k=11` or `k=15`, with classification error about `0.183` and weighted F1-score about `0.812`. The best normalized result is at `k=21`, with classification error about `0.240` and weighted F1-score about `0.754`. In this run, normalization did not improve performance.

Small values such as `k=1` can overfit because each prediction depends heavily on one nearby training sample. Larger values make the decision smoother, but very large values can underfit by ignoring local structure. Although normalization is often useful for k-NN, the original feature scales worked better for this specific real estate split.

## Dataset 2: Song popularity

The numeric song popularity target is converted to three classes. `song_name` is text, so it is removed from the k-NN feature matrix.

In [9]:
songs = pd.read_csv("song_data.csv")
songs.head()

,song_name,song_popularity,song_duration_ms,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,audio_mode,speechiness,tempo,time_signature,audio_valence
0,Boulevard of Broken Dreams,73,262333,0.005520,0.496,0.682,0.000029,8,0.0589,-4.095,1,0.0294,167.060,4,0.474
1,In The End,66,216933,0.010300,0.542,0.853,0.000000,3,0.1080,-6.407,0,0.0498,105.256,4,0.370
2,Seven Nation Army,76,231733,0.008170,0.737,0.463,0.447000,0,0.2550,-7.828,1,0.0792,123.881,4,0.324
3,By The Way,74,216933,0.026400,0.451,0.970,0.003550,0,0.1020,-4.938,1,0.1070,122.444,4,0.198
4,How You Remind Me,56,223826,0.000954,0.447,0.766,0.000000,10,0.1130,-5.065,1,0.0313,172.011,4,0.574


In [10]:
song_target = "song_popularity"

song_y = make_quantile_classes(songs[song_target])
song_X = songs.drop(columns=["song_name", song_target])

print("Class distribution:")
display(song_y.value_counts().sort_index())

print("Input features:")
display(song_X.head())

Class distribution:


song_popularity
low       6286
medium    6383
high      6166
Name: count, dtype: int64

Input features:


,song_duration_ms,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,audio_mode,speechiness,tempo,time_signature,audio_valence
0,262333,0.005520,0.496,0.682,0.000029,8,0.0589,-4.095,1,0.0294,167.060,4,0.474
1,216933,0.010300,0.542,0.853,0.000000,3,0.1080,-6.407,0,0.0498,105.256,4,0.370
2,231733,0.008170,0.737,0.463,0.447000,0,0.2550,-7.828,1,0.0792,123.881,4,0.324
3,216933,0.026400,0.451,0.970,0.003550,0,0.1020,-4.938,1,0.1070,122.444,4,0.198
4,223826,0.000954,0.447,0.766,0.000000,10,0.1130,-5.065,1,0.0313,172.011,4,0.574


## Task 1: Run k-NN on the Song Popularity Dataset

This section trains k-nearest neighbors classifiers on the song audio features using the binned `song_popularity` class label.

## Task 2 and Task 3: Song Metrics and Different k Values

The output table compares the predicted test labels with the original test labels for several values of `k`. It reports classification error, precision, recall, and F1-score.

In [11]:
song_results, song_best_k, song_best_model = evaluate_knn(song_X, song_y, normalize=False)
song_results

,k,classification_error,precision,recall,f1_score
0,1,0.458908,0.535401,0.541092,0.536669
1,3,0.537481,0.458596,0.462519,0.452032
2,5,0.563602,0.432769,0.436398,0.427625
3,7,0.572521,0.423815,0.427479,0.422667
4,9,0.581015,0.415115,0.418985,0.412759
5,11,0.587386,0.409322,0.412614,0.407073
6,15,0.596730,0.399964,0.403270,0.398875
7,21,0.598853,0.397084,0.401147,0.396413


### Task 3 Comment: Song Popularity k Values

For the song popularity dataset, the best result is at `k=1`. As `k` increases, classification error increases and weighted F1-score decreases. This means larger neighborhoods are not helpful for this dataset; averaging over more neighbors smooths away useful local patterns in the audio features.

## Task 4: Normalize the Song Dataset

This section repeats k-NN after applying `StandardScaler` to the input features.

In [12]:
song_scaled_results, song_scaled_best_k, song_scaled_best_model = evaluate_knn(song_X, song_y, normalize=True)
song_scaled_results

,k,classification_error,precision,recall,f1_score
0,1,0.440646,0.553946,0.559354,0.554878
1,3,0.498620,0.501518,0.501380,0.490497
2,5,0.526863,0.472172,0.473137,0.464210
3,7,0.534933,0.461571,0.465067,0.458138
4,9,0.539180,0.458477,0.460820,0.453574
5,11,0.543215,0.453576,0.456785,0.448853
6,15,0.553621,0.443592,0.446379,0.439000
7,21,0.560841,0.437406,0.439159,0.432150


## Song Normalized vs. Unnormalized Comparison

The table below places both versions side by side so the effect of normalization can be compared directly.

In [13]:
song_comparison = song_results.merge(
    song_scaled_results,
    on="k",
    suffixes=("_without_normalization", "_with_normalization")
)
song_comparison

,k,classification_error_without_normalization,precision_without_normalization,recall_without_normalization,f1_score_without_normalization,classification_error_with_normalization,precision_with_normalization,recall_with_normalization,f1_score_with_normalization
0,1,0.458908,0.535401,0.541092,0.536669,0.440646,0.553946,0.559354,0.554878
1,3,0.537481,0.458596,0.462519,0.452032,0.498620,0.501518,0.501380,0.490497
2,5,0.563602,0.432769,0.436398,0.427625,0.526863,0.472172,0.473137,0.464210
3,7,0.572521,0.423815,0.427479,0.422667,0.534933,0.461571,0.465067,0.458138
4,9,0.581015,0.415115,0.418985,0.412759,0.539180,0.458477,0.460820,0.453574
5,11,0.587386,0.409322,0.412614,0.407073,0.543215,0.453576,0.456785,0.448853
6,15,0.596730,0.399964,0.403270,0.398875,0.553621,0.443592,0.446379,0.439000
7,21,0.598853,0.397084,0.401147,0.396413,0.560841,0.437406,0.439159,0.432150


## Song Best Model Details

The classification reports and confusion matrices show the detailed test-set performance for the best unnormalized and normalized song popularity models.

In [14]:
print_best_model_details("Song popularity without normalization", song_best_k, song_best_model)
print_best_model_details("Song popularity with normalization", song_scaled_best_k, song_scaled_best_model)

Song popularity without normalization - best k: 1

Classification report:
              precision    recall  f1-score   support

        high       0.61      0.70      0.65      1541
         low       0.49      0.44      0.46      1572
      medium       0.51      0.49      0.50      1596

    accuracy                           0.54      4709
   macro avg       0.54      0.54      0.54      4709
weighted avg       0.54      0.54      0.54      4709

Confusion matrix:


,pred_high,pred_low,pred_medium
actual_high,1081,239,221
actual_low,358,686,528
actual_medium,332,483,781


Song popularity with normalization - best k: 1

Classification report:
              precision    recall  f1-score   support

        high       0.63      0.72      0.67      1541
         low       0.52      0.46      0.49      1572
      medium       0.51      0.50      0.51      1596

    accuracy                           0.56      4709
   macro avg       0.55      0.56      0.56      4709
weighted avg       0.55      0.56      0.55      4709

Confusion matrix:


,pred_high,pred_low,pred_medium
actual_high,1115,200,226
actual_low,324,717,531
actual_medium,331,463,802


### Song popularity comments

For the song popularity dataset, the best result is at `k=1` both before and after normalization. Without normalization, the classification error is about `0.459` and weighted F1-score is about `0.537`. With normalization, the classification error improves to about `0.441` and weighted F1-score improves to about `0.555`, so normalization gives a small improvement.

Performance becomes worse as `k` increases, which suggests that averaging over more neighbors smooths away useful local patterns in this dataset. Normalization is helpful here because features such as `song_duration_ms`, `tempo`, `loudness`, and fractional audio features are on very different scales.

## Overall conclusion

The best real estate model in this run is unnormalized k-NN with `k=11` or `k=15`. The best song popularity model is normalized k-NN with `k=1`. Normalization is not guaranteed to improve every dataset, but it is important to test for k-NN because the algorithm uses distances between feature values.

## Repeat Tasks 1-4: Logistic Regression and SVM

The same classification workflow is repeated for Logistic Regression and SVM. These models do not have a `k` value, so Task 3 is repeated by varying each model's own hyperparameters instead.

## Logistic Regression: Real Estate Dataset

This section trains Logistic Regression on the real estate classes, reports the same test-set metrics, varies `C`, and compares normalized vs. unnormalized input features.

In [ ]:
real_logistic_results, real_logistic_best_setting, real_logistic_best_model = evaluate_model_family(
    real_estate_X,
    real_estate_y,
    model_name="Logistic Regression",
    normalize=False
)
real_logistic_results

In [ ]:
real_logistic_scaled_results, real_logistic_scaled_best_setting, real_logistic_scaled_best_model = evaluate_model_family(
    real_estate_X,
    real_estate_y,
    model_name="Logistic Regression",
    normalize=True
)
real_logistic_scaled_results

In [ ]:
real_logistic_comparison = real_logistic_results.merge(
    real_logistic_scaled_results,
    on=["model", "setting"],
    suffixes=("_without_normalization", "_with_normalization")
)
real_logistic_comparison

### Task 3 Comment: Real Estate Logistic Regression

Varying `C` changes the strength of regularization. Smaller `C` means stronger regularization, while larger `C` allows the model to fit the training data more closely. Compare the error and F1-score in the table to decide which value of `C` works best.

## SVM: Real Estate Dataset

This section trains linear SVM models on the real estate classes, reports the same test-set metrics, varies `C`, and compares normalized vs. unnormalized input features.

In [ ]:
real_svm_results, real_svm_best_setting, real_svm_best_model = evaluate_model_family(
    real_estate_X,
    real_estate_y,
    model_name="SVM",
    normalize=False
)
real_svm_results

In [ ]:
real_svm_scaled_results, real_svm_scaled_best_setting, real_svm_scaled_best_model = evaluate_model_family(
    real_estate_X,
    real_estate_y,
    model_name="SVM",
    normalize=True
)
real_svm_scaled_results

In [ ]:
real_svm_comparison = real_svm_results.merge(
    real_svm_scaled_results,
    on=["model", "setting"],
    suffixes=("_without_normalization", "_with_normalization")
)
real_svm_comparison

### Task 3 Comment: Real Estate SVM

For linear SVM, varying `C` changes the strength of regularization. Smaller `C` gives stronger regularization, while larger `C` allows the margin to fit the training data more closely. Compare the table to see which value gives the best test-set F1-score.

## Logistic Regression: Song Popularity Dataset

This section trains Logistic Regression on the song popularity classes, reports the same test-set metrics, varies `C`, and compares normalized vs. unnormalized input features.

In [ ]:
song_logistic_results, song_logistic_best_setting, song_logistic_best_model = evaluate_model_family(
    song_X,
    song_y,
    model_name="Logistic Regression",
    normalize=False
)
song_logistic_results

In [ ]:
song_logistic_scaled_results, song_logistic_scaled_best_setting, song_logistic_scaled_best_model = evaluate_model_family(
    song_X,
    song_y,
    model_name="Logistic Regression",
    normalize=True
)
song_logistic_scaled_results

In [ ]:
song_logistic_comparison = song_logistic_results.merge(
    song_logistic_scaled_results,
    on=["model", "setting"],
    suffixes=("_without_normalization", "_with_normalization")
)
song_logistic_comparison

### Task 3 Comment: Song Logistic Regression

For song popularity, compare `C=0.1`, `C=1`, and `C=10`. If the scores are very close, the model is not very sensitive to this hyperparameter. Normalization is usually expected to help Logistic Regression because it makes the feature scales comparable.

## SVM: Song Popularity Dataset

This section trains linear SVM models on the song popularity classes, reports the same test-set metrics, varies `C`, and compares normalized vs. unnormalized input features.

In [ ]:
song_svm_results, song_svm_best_setting, song_svm_best_model = evaluate_model_family(
    song_X,
    song_y,
    model_name="SVM",
    normalize=False
)
song_svm_results

In [ ]:
song_svm_scaled_results, song_svm_scaled_best_setting, song_svm_scaled_best_model = evaluate_model_family(
    song_X,
    song_y,
    model_name="SVM",
    normalize=True
)
song_svm_scaled_results

In [ ]:
song_svm_comparison = song_svm_results.merge(
    song_svm_scaled_results,
    on=["model", "setting"],
    suffixes=("_without_normalization", "_with_normalization")
)
song_svm_comparison

### Task 3 Comment: Song SVM

For linear SVM, the comparison shows which `C` value works best. If the normalized scores are higher, it means the model benefits from feature scaling before fitting the separating hyperplanes.

In [ ]:
summary_rows = []

for dataset_name, result_sets in {
    "Real estate": [
        ("k-NN", real_results, real_scaled_results),
        ("Logistic Regression", real_logistic_results, real_logistic_scaled_results),
        ("SVM", real_svm_results, real_svm_scaled_results),
    ],
    "Song popularity": [
        ("k-NN", song_results, song_scaled_results),
        ("Logistic Regression", song_logistic_results, song_logistic_scaled_results),
        ("SVM", song_svm_results, song_svm_scaled_results),
    ]
}.items():
    for model_name, plain_results, scaled_results in result_sets:
        plain_best = plain_results.sort_values(["f1_score", "classification_error"], ascending=[False, True]).iloc[0]
        scaled_best = scaled_results.sort_values(["f1_score", "classification_error"], ascending=[False, True]).iloc[0]
        summary_rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "best_without_normalization": plain_best.get("setting", plain_best.get("k")),
            "f1_without_normalization": plain_best["f1_score"],
            "error_without_normalization": plain_best["classification_error"],
            "best_with_normalization": scaled_best.get("setting", scaled_best.get("k")),
            "f1_with_normalization": scaled_best["f1_score"],
            "error_with_normalization": scaled_best["classification_error"],
            "normalization_better": scaled_best["f1_score"] > plain_best["f1_score"]
        })

model_summary = pd.DataFrame(summary_rows)
model_summary

## Overall Comments for Logistic Regression and SVM

Use the summary table above to compare the best unnormalized and normalized result for each model. For Logistic Regression and SVM, normalization is often important because both methods are affected by feature scale. The best model is the one with the lowest classification error and the highest weighted F1-score on the test data.